In [1]:
from scipy.stats import qmc
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.gaussian_process.kernels import Matern

In [2]:
#Function 8
#Extracting updated data and turning it into a pandas dataframe
data = pd.read_csv('Data/Week 6 - Function 8.csv') 
columns = ['Input 1', 'Input 2','Input 3', 'Input 4','Input 5','Input 6','Input 7', 'Input 8', 'Outputs']
#Remove columns with NaN values
data = data.dropna(axis = 1)
data.columns = columns
#Add Week 7's data to our pandas dataframe
new_data7 = np.array([0.006676, 0.142850, 0.014612, 0.008493, 0.013195, 0.029041, 0.146052, 0.709045
                     ,9.384219448563])
data.loc[len(data)] = new_data7
#Add Week 8's data to our pandas dataframe
new_data8 = np.array([0.014396, 0.156572, 0.036172, 0.303020, 0.785780, 0.313016, 0.213195, 0.543895
                     ,9.8997474044235])
data.loc[len(data)] = new_data8
#Add Week 9's data to our pandas dataframe
new_data9 = np.array([0.077029, 0.211423, 0.024938, 0.360429, 0.412992, 0.719907, 0.020932, 0.647577
                     ,9.7301737067941])
data.loc[len(data)] = new_data9
#Add Week 10's data to our pandas dataframe
new_data10 = np.array([0.042445, 0.051647, 0.050686, 0.076565, 0.916977, 0.600690, 0.111062, 0.775604
                     ,9.9235528027939])
data.loc[len(data)] = new_data10
#Add Week 11's data to our pandas dataframe
new_data11 = np.array([0.022224, 0.593235, 0.013522, 0.089016, 0.932122, 0.236391, 0.200148, 0.796551
                     ,9.664943011023901])
data.loc[len(data)] = new_data11
data

,Input 1,Input 2,Input 3,Input 4,Input 5,Input 6,Input 7,Input 8,Outputs
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [7]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4', 'Input 5', 'Input 6',
                  'Input 7', 'Input 8']])
Y = np.array(data[['Outputs']])


In [8]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 8
n = 900000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Set radius of hypercube
delta = 0.2
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)


In [9]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)

gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [12]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find next point that maximises acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.09192  0.203748 0.09847  0.166686 0.732304 0.51622  0.225742 0.703709]


In [3]:
#Week 12
#Add new data
new_data = np.array([0.091920, 0.203748, 0.098470, 0.166686, 0.732304, 0.516220, 0.225742, 0.703709,9.9887644149959])
data.loc[len(data)] = new_data
data 

,Input 1,Input 2,Input 3,Input 4,Input 5,Input 6,Input 7,Input 8,Outputs
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [4]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4', 'Input 5', 'Input 6',
                  'Input 7', 'Input 8']])
Y = np.array(data[['Outputs']])

In [11]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 8
n = 900000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Increase radius of hypercube by 10% since there was an improvement
delta = 0.2*1.1
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.09192  0.203748 0.09847  0.166686 0.732304 0.51622  0.225742 0.703709]
Upper bound [0.31192  0.423748 0.31847  0.386686 0.952304 0.73622  0.445742 0.923709]
lower bound [-0.12808  -0.016252 -0.12153  -0.053314  0.512304  0.29622   0.005742
  0.483709]


In [12]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)

gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [14]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find next point that maximises acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.085097 0.20527  0.149789 0.148856 0.665315 0.515868 0.211622 0.68971 ]


In [4]:
#Week 12
#Add new data
new_data = np.array([0.085097, 0.205270, 0.149789, 0.148856, 0.665315, 0.515868, 0.211622, 0.689710,9.9849281577685])
data.loc[len(data)] = new_data
data 

,Input 1,Input 2,Input 3,Input 4,Input 5,Input 6,Input 7,Input 8,Outputs
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [5]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4', 'Input 5', 'Input 6',
                  'Input 7', 'Input 8']])
Y = np.array(data[['Outputs']])

In [7]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 8
n = 900000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Decrease radius of hypercube by 10% since there was an improvement
delta = 0.2*1.1*0.9
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.09192  0.203748 0.09847  0.166686 0.732304 0.51622  0.225742 0.703709]
Upper bound [0.28992  0.401748 0.29647  0.364686 0.930304 0.71422  0.423742 0.901709]
lower bound [-0.10608   0.005748 -0.09953  -0.031314  0.534304  0.31822   0.027742
  0.505709]


In [8]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)

gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [9]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find next point that maximises acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.091143 0.246147 0.118623 0.14505  0.699832 0.503704 0.187565 0.678935]
